In [ ]:
from importlib import reload
from pathlib import Path
from typing import Literal

import polars as pl

from ctm_ai_eval.utils import plots

plots.set_template()
reload(plots)
MODE: Literal["query", "paraphrase", "verbatim"] = "query"

RESULT_ROOT = Path("../tmp/results/")
available_experiments = [p.stem for p in RESULT_ROOT.glob("*/")]
print(f"{available_experiments=}")


## Chunker


In [ ]:
df_avg = pl.read_ndjson(RESULT_ROOT / "haystack-chunkers" / "averages.ndjson")
df_avg = df_avg.filter(pl.col("retriever") == "faiss_nomic-embed-text")
df_avg = df_avg.with_columns(pl.col("chunker").str.split("(").list.first().alias("chunker_type"))
reload(plots)
plots.scatter_recall_rr(df_avg, "needle", k=5)

plots.multi_metric_bar(
    df_avg.filter(pl.col("needle") == MODE).sort("rr"),
    x="chunker_type",
).update_layout(title=f"metrics for '{MODE}'").show()

In [ ]:
import plotly.graph_objects as go

df = df_avg.filter(pl.col("needle") == "query")
RECALL_COLS = [c for c in df.columns if c.startswith("recall")]
K_VALUES = [int(c.removeprefix("recall@")) for c in RECALL_COLS]

# one colour per chunker_type
types = df["chunker_type"].unique().to_list()
colors = [
    "#378ADD",
    "#D85A30",
    "#D4537E",
    "#7F77DD",
    "#BA7517",
    "#E24B4A",
    "#639922",
]
color_map = {t: colors[i % len(colors)] for i, t in enumerate(types)}

fig = go.Figure()

for chunker_type, group in df.group_by("chunker_type"):
    chunker_type = chunker_type[0]
    color = color_map[chunker_type]
    means = [group[col].mean() for col in RECALL_COLS]

    # --- individual runs (faint lines, no legend entry) ---
    for row in group.iter_rows(named=True):
        fig.add_trace(
            go.Scatter(
                x=K_VALUES,
                y=[row[col] for col in RECALL_COLS],
                mode="lines",
                line=dict(color=color, width=2),
                opacity=0.5,
                showlegend=False,
                name=row["chunker"],
                hovertemplate="%{fullData.name}<extra></extra>",
            )
        )


fig.update_layout(
    xaxis=dict(
        title="k",
        tickvals=K_VALUES,
        ticktext=["@1", "@5", "@10"],
    ),
    yaxis=dict(title="recall", range=[0, 1]),
    title="Recall for each chunker setup",
)

fig.show()

## Retriever


In [ ]:
df_avg = pl.read_ndjson(RESULT_ROOT / "haystack_retrievers" / "averages.ndjson")
df_avg = df_avg.with_columns(pl.col("chunker").str.split("(").list.first().alias("chunker_type"))
reload(plots)
plots.scatter_recall_rr(df_avg.filter(pl.col("needle") == MODE), "retriever", k=5)


plots.multi_metric_bar(
    df_avg.filter(pl.col("needle") == MODE).sort("rr"),
    x="retriever",
).update_layout(title=f"metrics for '{MODE}'").show()